In [6]:
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

In [9]:
dataset_all = pd.read_csv('../data/01_raw/bger-2024-1.csv')

In [10]:
# Filter for criminal cases only
criminal_cases = dataset_all[dataset_all['proc_type'] == 'Criminal'].copy()

# Create binary target: granted = 1, else = 0
criminal_cases['outcome_binary'] = (criminal_cases['outcome'] == 'granted').astype(int)

# Select features (numeric and categorical)
feature_cols = ['year', 'merged_cases', 'n_judges', 'proc_duration', 'app_represented', 'resp_represented']
criminal_cases = criminal_cases[feature_cols + ['outcome_binary']].dropna()

# Encode boolean/categorical columns
le_dict = {}
for col in ['merged_cases', 'app_represented', 'resp_represented']:
    if col in criminal_cases.columns:
        le = LabelEncoder()
        criminal_cases[col] = le.fit_transform(criminal_cases[col].astype(str))
        le_dict[col] = le

# Prepare X and y
X = criminal_cases[feature_cols]
y = criminal_cases['outcome_binary']



In [19]:
# Train-test split
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train LightGBM model
model = lgb.LGBMClassifier(random_state=42, verbose=-1)

# make the classifier more coplex, it is severly underfitting the data, so we will increase the number of estimators and the max depth
model = lgb.LGBMClassifier(random_state=42, verbose=-1, n_estimators=1000, max_depth=15)

model.fit(X_train, y_train)

# Evaluate
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print(f"Train accuracy: {train_score:.4f}")
print(f'Mean of predictions on training set: {model.predict(X_train).mean():.4f}')
print(f"Test accuracy: {test_score:.4f}")

Train accuracy: 0.9336
Mean of predictions on training set: 0.0336
Test accuracy: 0.8994


In [ ]:
# export model to some pickle format so that it can be used in the swarm
model.save_model('../models/lgbm_criminal_cases_model.txt')

np.float64(0.09305733650851804)

In [14]:
model.predict(X_train).mean()

np.float64(0.007281553398058253)